In [134]:
#import the modules 
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
import os
import sys
import re
import json
import openpyxl
import warnings
warnings.filterwarnings('ignore')


### Loading our dataset

Let's load our ESG dataser from Excel to Pandas dataframe

In [135]:
# Load the data from the Excel file
data_file_path = "../data/ESG_Dataset.xlsx"
df = pd.read_excel(data_file_path, sheet_name="Emissions Data")


### Data Exploration
We will be check for head(), tail(), info(), shape, duplicates, missing values, Column names, etc. We will also examine the dtypes for the columns in Dataframe.

In [136]:
df.head()

,Site Name,Year,Month,Scope,Emission Category,Emission Source,Activity Value,Activity Unit,Emission Factor,Emission Value (tCO2e),DEFRA_Emission Value (tCO2e),Collected By,Evidence Type,Calculation
0,M-1,2024,January,Scope-1,Fuel Combustion,Coal,359747.14,kg,2.41 tCO2e/ton,NaN,NaN,Facility Manager,Supplier Invoice,NaN
1,M-1,2024,January,Scope-1,Fuel Combustion,Diesel,5806.23,Liters,2.68 kg CO2e/L,NaN,NaN,Facility Manager,Invoice,NaN
2,M-1,2024,January,Scope-2,Purchased Electricity,Grid Electricity (Location-based),53895.95,kWh,0.727 kg CO2e/kWh,NaN,NaN,Site Supervisor,Utility Bill,NaN
3,M-1,2024,February,Scope-1,Fuel Combustion,Coal,307297.23,kg,2.41 tCO2e/ton,NaN,NaN,Maintenance Team,Supplier Invoice,NaN
4,M-1,2024,February,Scope-1,Fuel Combustion,Diesel,5045.91,Liters,2.68 kg CO2e/L,NaN,NaN,Energy Manager,Invoice,NaN


In [137]:
df.tail()

,Site Name,Year,Month,Scope,Emission Category,Emission Source,Activity Value,Activity Unit,Emission Factor,Emission Value (tCO2e),DEFRA_Emission Value (tCO2e),Collected By,Evidence Type,Calculation
223,W-1,2024,October,Scope-2,Purchased Electricity,Grid Electricity (Location-based),12856.08,kWh,0.727 kg CO2e/kWh,NaN,NaN,Energy Manager,Utility Bill,NaN
224,W-1,2024,November,Scope-1,Fuel Combustion,Diesel,750.85,Liters,2.68 kg CO2e/L,NaN,NaN,Maintenance Team,Invoice,NaN
225,W-1,2024,November,Scope-2,Purchased Electricity,Grid Electricity (Location-based),13233.45,kWh,0.727 kg CO2e/kWh,NaN,NaN,Facility Manager,Utility Bill,NaN
226,W-1,2024,December,Scope-1,Fuel Combustion,Diesel,779.47,Liters,2.68 kg CO2e/L,NaN,NaN,Facility Manager,Invoice,NaN
227,W-1,2024,December,Scope-2,Purchased Electricity,Grid Electricity (Location-based),14790.62,kWh,0.727 kg CO2e/kWh,NaN,NaN,Energy Manager,Utility Bill,NaN


### Shape of the dataset

In [138]:
df.shape

(228, 14)

Our dataset has 228 and 14 columns 
  

### Columns in our dataset

In [139]:
df.columns.tolist()

['Site Name',
 'Year',
 'Month',
 'Scope',
 'Emission Category',
 'Emission Source',
 'Activity Value',
 'Activity Unit',
 'Emission Factor',
 'Emission Value (tCO2e)',
 'DEFRA_Emission Value (tCO2e)',
 'Collected By',
 'Evidence Type',
 'Calculation']

In [140]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228 entries, 0 to 227
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Site Name                     228 non-null    object 
 1   Year                          228 non-null    int64  
 2   Month                         228 non-null    object 
 3   Scope                         228 non-null    object 
 4   Emission Category             228 non-null    object 
 5   Emission Source               228 non-null    object 
 6   Activity Value                228 non-null    float64
 7   Activity Unit                 228 non-null    object 
 8   Emission Factor               228 non-null    object 
 9   Emission Value (tCO2e)        0 non-null      float64
 10  DEFRA_Emission Value (tCO2e)  0 non-null      float64
 11  Collected By                  228 non-null    object 
 12  Evidence Type                 228 non-null    object 
 13  Calcu

As noticed above, `Year` is *int*  and `Month` is *str* object. we have to convert these 2 columns into *datetime* format. We might also want to split and convert the `Scope` column into a *int* value and use onehot encoding later on. The numeric columns `Emission Factor` is  a string and has units. We have to extract the numeric value out of this column and create new columns with just the values. Also, you can see `Emission Value (tCO2e)` , `DEFRA_Emission Value (tCO2e)` and `Calculation` seems to be empty. We have to decide how to calculate the values or drop the columns if they are not relevant.

### Check for missing values

We already noted the missing values with df.info(), but lets check it anyway to be sure.

In [141]:
#check for number of null values in each column
df.isnull().sum()

Site Name                         0
Year                              0
Month                             0
Scope                             0
Emission Category                 0
Emission Source                   0
Activity Value                    0
Activity Unit                     0
Emission Factor                   0
Emission Value (tCO2e)          228
DEFRA_Emission Value (tCO2e)    228
Collected By                      0
Evidence Type                     0
Calculation                     228
dtype: int64

### Check for duplicates 

In [142]:
# check for duplicate rows
df.duplicated().sum()

np.int64(0)

No duplicate rows found. 

### Dataset Pre-process
 We will prepare the data for anaylysis. The pre-process items identified in the Data Exploration phase will be implemented here. 

1. Extract the numeric values in column `Emission Facotr`  and add the split valyes into 2 new columns `Emission Factor Value` as float64 and `Emission Factor Unit` as str

In [143]:

df['Emission Factor Value'] = df['Emission Factor'].astype(str).str.split().str[0].astype(float)
df['Emission Factor Unit'] = df['Emission Factor'].astype(str).str.split().str[1]
df.head()


,Site Name,Year,Month,Scope,Emission Category,Emission Source,Activity Value,Activity Unit,Emission Factor,Emission Value (tCO2e),DEFRA_Emission Value (tCO2e),Collected By,Evidence Type,Calculation,Emission Factor Value,Emission Factor Unit
0,M-1,2024,January,Scope-1,Fuel Combustion,Coal,359747.14,kg,2.41 tCO2e/ton,NaN,NaN,Facility Manager,Supplier Invoice,NaN,2.410,tCO2e/ton
1,M-1,2024,January,Scope-1,Fuel Combustion,Diesel,5806.23,Liters,2.68 kg CO2e/L,NaN,NaN,Facility Manager,Invoice,NaN,2.680,kg
2,M-1,2024,January,Scope-2,Purchased Electricity,Grid Electricity (Location-based),53895.95,kWh,0.727 kg CO2e/kWh,NaN,NaN,Site Supervisor,Utility Bill,NaN,0.727,kg
3,M-1,2024,February,Scope-1,Fuel Combustion,Coal,307297.23,kg,2.41 tCO2e/ton,NaN,NaN,Maintenance Team,Supplier Invoice,NaN,2.410,tCO2e/ton
4,M-1,2024,February,Scope-1,Fuel Combustion,Diesel,5045.91,Liters,2.68 kg CO2e/L,NaN,NaN,Energy Manager,Invoice,NaN,2.680,kg


2. We will not need `DEFRA_Emission Value (tCO2e)` and `Calculation` values for this report. DEFRA Emission values are a measure used by UK Government which have diffrent values for the Emission factor based on the activity and source of fuel. Lets drop the 2 columns. We can also drop the Emission Factor column now as we have 2 new columns to indicate that. 

In [144]:
# Drop the original 'Emission Factor' column, since we have split it into two new columns 
# Also drop the DEFRA_Emission Value (tCO2e) column  and Calculation column as it is not needed for analysis

esg_df  = df.copy()

esg_df.drop(columns=['Emission Factor', 'DEFRA_Emission Value (tCO2e)', 'Calculation'], inplace=True)



3. Add a`Date` column to  convert the `Year` and `Month` column to datetime format and. Then add one more column `Year_Month` so the format is **2024-01** for example. 

In [145]:
#Add a`Date` column to  convert the `Year` and `Month` column to datetime format and. Then add one more column `Year_Month` so the format is **2024-01** for example. 
esg_df['Date'] = pd.to_datetime(df['Year'].astype(str) + ' ' + df['Month'].str.strip(), format='%Y %B')


In [146]:
#Then add one more column `Year_Month` so the format is **2024-01** for example. 
esg_df['Year_Month'] = esg_df['Date'].dt.to_period('M')

In [147]:
esg_df.head()

,Site Name,Year,Month,Scope,Emission Category,Emission Source,Activity Value,Activity Unit,Emission Value (tCO2e),Collected By,Evidence Type,Emission Factor Value,Emission Factor Unit,Date,Year_Month
0,M-1,2024,January,Scope-1,Fuel Combustion,Coal,359747.14,kg,NaN,Facility Manager,Supplier Invoice,2.410,tCO2e/ton,2024-01-01,2024-01
1,M-1,2024,January,Scope-1,Fuel Combustion,Diesel,5806.23,Liters,NaN,Facility Manager,Invoice,2.680,kg,2024-01-01,2024-01
2,M-1,2024,January,Scope-2,Purchased Electricity,Grid Electricity (Location-based),53895.95,kWh,NaN,Site Supervisor,Utility Bill,0.727,kg,2024-01-01,2024-01
3,M-1,2024,February,Scope-1,Fuel Combustion,Coal,307297.23,kg,NaN,Maintenance Team,Supplier Invoice,2.410,tCO2e/ton,2024-02-01,2024-02
4,M-1,2024,February,Scope-1,Fuel Combustion,Diesel,5045.91,Liters,NaN,Energy Manager,Invoice,2.680,kg,2024-02-01,2024-02


4. Drop the `Year`, `Month` and `Date` columns as we now have `Year_Month` column

In [148]:
# Drop the Year, Month and Date columns as they are no longer needed since we have the Year_Month column
esg_df = esg_df.drop(['Year', 'Month', 'Date'], axis=1)
esg_df.head()


,Site Name,Scope,Emission Category,Emission Source,Activity Value,Activity Unit,Emission Value (tCO2e),Collected By,Evidence Type,Emission Factor Value,Emission Factor Unit,Year_Month
0,M-1,Scope-1,Fuel Combustion,Coal,359747.14,kg,NaN,Facility Manager,Supplier Invoice,2.410,tCO2e/ton,2024-01
1,M-1,Scope-1,Fuel Combustion,Diesel,5806.23,Liters,NaN,Facility Manager,Invoice,2.680,kg,2024-01
2,M-1,Scope-2,Purchased Electricity,Grid Electricity (Location-based),53895.95,kWh,NaN,Site Supervisor,Utility Bill,0.727,kg,2024-01
3,M-1,Scope-1,Fuel Combustion,Coal,307297.23,kg,NaN,Maintenance Team,Supplier Invoice,2.410,tCO2e/ton,2024-02
4,M-1,Scope-1,Fuel Combustion,Diesel,5045.91,Liters,NaN,Energy Manager,Invoice,2.680,kg,2024-02


5. Modify the `Scope` column to be an *int* with numeric values only. This may be useful for dummy variables and onehot encoding later on for further analysis



In [149]:
#Modify the 'Scope' column to be an *int* with numeric values only. This may be useful for dummy variables and onehot encoding later on for further analysis
esg_df['Scope'] = esg_df['Scope'].str.extract('(\d+)').astype(int)
esg_df.head()

,Site Name,Scope,Emission Category,Emission Source,Activity Value,Activity Unit,Emission Value (tCO2e),Collected By,Evidence Type,Emission Factor Value,Emission Factor Unit,Year_Month
0,M-1,1,Fuel Combustion,Coal,359747.14,kg,NaN,Facility Manager,Supplier Invoice,2.410,tCO2e/ton,2024-01
1,M-1,1,Fuel Combustion,Diesel,5806.23,Liters,NaN,Facility Manager,Invoice,2.680,kg,2024-01
2,M-1,2,Purchased Electricity,Grid Electricity (Location-based),53895.95,kWh,NaN,Site Supervisor,Utility Bill,0.727,kg,2024-01
3,M-1,1,Fuel Combustion,Coal,307297.23,kg,NaN,Maintenance Team,Supplier Invoice,2.410,tCO2e/ton,2024-02
4,M-1,1,Fuel Combustion,Diesel,5045.91,Liters,NaN,Energy Manager,Invoice,2.680,kg,2024-02


6. Rearrange the columns so it's easier

In [150]:
#reorder the columns to have a more logical order for analysis
new_order = [
    'Site Name', 'Year_Month', 'Scope', 
    'Emission Category', 'Emission Source', 
    'Activity Value', 'Activity Unit', 'Emission Factor Value', 'Emission Factor Unit',
    'Emission Value (tCO2e)', 'Collected By', 'Evidence Type'
]

esg_df = esg_df[new_order]

esg_df.head()


,Site Name,Year_Month,Scope,Emission Category,Emission Source,Activity Value,Activity Unit,Emission Factor Value,Emission Factor Unit,Emission Value (tCO2e),Collected By,Evidence Type
0,M-1,2024-01,1,Fuel Combustion,Coal,359747.14,kg,2.410,tCO2e/ton,NaN,Facility Manager,Supplier Invoice
1,M-1,2024-01,1,Fuel Combustion,Diesel,5806.23,Liters,2.680,kg,NaN,Facility Manager,Invoice
2,M-1,2024-01,2,Purchased Electricity,Grid Electricity (Location-based),53895.95,kWh,0.727,kg,NaN,Site Supervisor,Utility Bill
3,M-1,2024-02,1,Fuel Combustion,Coal,307297.23,kg,2.410,tCO2e/ton,NaN,Maintenance Team,Supplier Invoice
4,M-1,2024-02,1,Fuel Combustion,Diesel,5045.91,Liters,2.680,kg,NaN,Energy Manager,Invoice


7. Calculate the `Emission Value (tCO2e)` using the formula  Activity_value * Emmission_Factor = Emission Value. Be careful to look a the different unit measure of each type of activity and the Emmission facor units. The Emission Value is measure in metric tonnes of CO2e


In [151]:
# Calculate 'Emission Value (tCO2e)'
# All three sources require dividing by 1,000:
# - Coal: (Activity_kg / 1000) * 2.41
# - Diesel: (Activity_L * 2.68) / 1000
# - Grid Electricity: (Activity_kWh * 0.727) / 1000
esg_df['Emission Value (tCO2e)'] = (esg_df['Activity Value'] * esg_df['Emission Factor Value']) / 1000

# Round to 2 decimal places for clean reporting
esg_df['Emission Value (tCO2e)'] = np.round(esg_df['Emission Value (tCO2e)'], 2)

esg_df.tail()


,Site Name,Year_Month,Scope,Emission Category,Emission Source,Activity Value,Activity Unit,Emission Factor Value,Emission Factor Unit,Emission Value (tCO2e),Collected By,Evidence Type
223,W-1,2024-10,2,Purchased Electricity,Grid Electricity (Location-based),12856.08,kWh,0.727,kg,9.35,Energy Manager,Utility Bill
224,W-1,2024-11,1,Fuel Combustion,Diesel,750.85,Liters,2.680,kg,2.01,Maintenance Team,Invoice
225,W-1,2024-11,2,Purchased Electricity,Grid Electricity (Location-based),13233.45,kWh,0.727,kg,9.62,Facility Manager,Utility Bill
226,W-1,2024-12,1,Fuel Combustion,Diesel,779.47,Liters,2.680,kg,2.09,Facility Manager,Invoice
227,W-1,2024-12,2,Purchased Electricity,Grid Electricity (Location-based),14790.62,kWh,0.727,kg,10.75,Energy Manager,Utility Bill


In [152]:
#save this esg_df to a csv file in the data folder as esg_df_cleaned.csv
output_file_path = "../data/esg_df_cleaned.csv"
esg_df.to_csv(output_file_path, index=False)

## Plotly visualizations

In [159]:
#import plotly.express as px imported at the top of the file.

# 1. Creating some shortcuts for column names to make the code more readable and maintainable.

EMISSIONS = "Emission Value (tCO2e)"

required_columns = [
    "Site Name",
    "Year_Month",
    "Scope",
    "Emission Source",
    EMISSIONS,
]

missing_columns = set(required_columns) - set(esg_df.columns)

if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

# Use text labels so Plotly treats scope as a category.
esg_df["Scope Label"] = "Scope " + esg_df["Scope"].astype(str)

px.defaults.template = "plotly_white"

scope_colors = {
    "Scope 1": "#2563EB",
    "Scope 2": "#F59E0B",
}

In [160]:
# 2. Basic analysis

#calculating total emissions for the entire dataset

total_emissions = esg_df[EMISSIONS].sum()


print(f"Rows: {len(esg_df):,}")
print(f"Sites: {esg_df['Site Name'].nunique()}")
print(
    f"Period: {esg_df['Year_Month'].min().strftime('%b %Y')}"
    f" to {esg_df['Year_Month'].max().strftime('%b %Y')}"
)
print(f"Total emissions: {total_emissions} tCO2e")

#calculate emissions by site and sort in descending order
site_summary = (
    esg_df.groupby("Site Name")[EMISSIONS]
    .sum()
    .sort_values(ascending=False)
    .to_frame("Total emissions (tCO2e)")
)
site_summary["Share (%)"] = (
    site_summary["Total emissions (tCO2e)"]
    .div(total_emissions)
    .mul(100)
)

print("\nEmissions by site:")
print(site_summary.round(2).to_string())

#calculate emissions by scope and sort in descending order

print("\nEmissions by scope:")
print(
    esg_df.groupby("Scope Label")[EMISSIONS]
    .agg(["sum", "count"])
    .rename(columns={"sum": "Total tCO2e", "count": "Records"})
    .round(2)
    .to_string()
)

Rows: 228
Sites: 7
Period: Jan 2024 to Dec 2024
Total emissions: 37460.380000000005 tCO2e

Emissions by site:
           Total emissions (tCO2e)  Share (%)
Site Name                                    
M-1                        9465.41      25.27
M-2                        8975.12      23.96
M-3                        7127.61      19.03
M-4                        6806.48      18.17
M-5                        4705.35      12.56
O-1                         235.81       0.63
W-1                         144.60       0.39

Emissions by scope:
             Total tCO2e  Records
Scope Label                      
Scope 1         35396.86      144
Scope 2          2063.52       84


In [161]:
# Chart 1: Monthly emissions by scope



monthly_scope = (
    esg_df.groupby(["Year_Month", "Scope Label"], as_index=False)[EMISSIONS]
    .sum()
    .sort_values(["Scope Label", "Year_Month"])
)

# Converting the  Pandas Period object(Year_Month column) to standard datetime64[ns] timestamps for Plotly
monthly_scope["Year_Month"] = monthly_scope["Year_Month"].dt.to_timestamp()

fig1 = px.line(
    monthly_scope,
    x="Year_Month",
    y=EMISSIONS,
    color="Scope Label",
    markers=True,
    color_discrete_map=scope_colors,
    title="Monthly Emissions by Scope",
    labels={
        "Year_Month": "Month",
        EMISSIONS: "Emissions (tCO2e)",
        "Scope Label": "Scope",
    },
)

fig1.update_layout(hovermode="x unified")
fig1.update_xaxes(dtick="M1", tickformat="%b %Y")
fig1.update_yaxes(tickformat=",.0f", rangemode="tozero")

In [162]:
# Chart 2: Site emissions, stacked by scope

site_scope = (
    esg_df.groupby(["Site Name", "Scope Label"], as_index=False)[EMISSIONS]
    .sum()
)

site_order = site_summary.index.tolist()

fig2 = px.bar(
    site_scope,
    x="Site Name",
    y=EMISSIONS,
    color="Scope Label",
    barmode="stack",
    category_orders={"Site Name": site_order},
    color_discrete_map=scope_colors,
    title="Total Emissions by Site and Scope",
    labels={
        EMISSIONS: "Emissions (tCO2e)",
        "Scope Label": "Scope",
    },
)

fig2.update_yaxes(tickformat=",.0f")
fig2.update_traces(
    hovertemplate=(
        "Site: %{x}<br>"
        "Emissions: %{y:,.2f} tCO2e"
        "<extra>%{fullData.name}</extra>"
    )
)

In [175]:
# Chart 3: Emissions by source

source_summary = (
    esg_df.groupby("Emission Source", as_index=False)[EMISSIONS]
    .sum()
    .sort_values(EMISSIONS, ascending=False)
)

fig3 = px.pie(
    source_summary,
    names="Emission Source",
    values=EMISSIONS,
    hole=0.55,
    title="Share of Emissions by Source",
    color_discrete_sequence=px.colors.qualitative.Safe,
)

fig3.update_traces(
    textposition="outside",
    textinfo="percent+label",
    hovertemplate=(
        "%{label}<br>"
        "Emissions: %{value:,.2f} tCO2e<br>"
        "Share: %{percent}"
        "<extra></extra>"
    ),
)

In [172]:
# Chart 4: Site-by-month emissions heatmap

site_month = esg_df.pivot_table(
    index="Site Name",
    columns="Year_Month",
    values=EMISSIONS,
    aggfunc="sum",
)

# Display every month in the observed date range.
# Leave missing site-month combinations as gaps, not assumed zeros.
all_months = pd.period_range(
    esg_df["Year_Month"].min(),
    esg_df["Year_Month"].max(),
    freq="M",
)
site_month = site_month.reindex(
    index=site_order,
    columns=all_months,
)
site_month.columns = site_month.columns.strftime("%b %Y")

fig4 = px.imshow(
    site_month,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    text_auto=",.0f",
    title="Monthly Emissions Heatmap by Site",
    labels={
        "x": "Month",
        "y": "Site",
        "color": "Emissions (tCO2e)",
    },
)

fig4.update_xaxes(side="bottom")
fig4.update_traces(
    hovertemplate=(
        "Site: %{y}<br>"
        "Month: %{x}<br>"
        "Emissions: %{z:,.2f} tCO2e"
        "<extra></extra>"
    ),
    hoverongaps=False,
)

In [176]:
# --------------------------------------------------
# . Display the four interactive charts
# --------------------------------------------------
for fig in [fig1, fig2, fig3, fig4]:
    fig.update_layout(
        font=dict(family="Arial", size=13),
        title_x=0.02,
        margin=dict(t=80, b=70, l=70, r=70),
        height=500,
    )
    fig.show()